In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import scanpy as sc
import pandas as pd
import pickle as pkl


from pathlib import Path
from tqdm.auto import tqdm
from wppkg.dl import hf_download
from wppkg import guess_is_lognorm
from unimol_tools import UniMolRepr
from wppkg import split_anndata_on_celltype
from descope.tokenizer import tokenize_adata_to_hf_dataset_for_rna
from descope.utils import UniformFeatureForAnnData, DuplicatedFeatureHandling

Data was downloaded from Hetzel et Al. [[1]](https://openreview.net/forum?id=vRrFVHxFiXJ), [`sciplex_complete_middle_subset.h5ad`](https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad) and processed as detailed below.

[[1] Hetzel, Leon, Simon Böhm, Niki Kilbertus, Stephan Günnemann, Mohammad Lotfollahi, and Fabian Theis. "Predicting Cellular Responses to Novel Drug Perturbations at a Single-Cell Resolution" Advances in Neural Information Processing Systems (2022).](https://papers.nips.cc/paper_files/paper/2022/hash/aa933b5abc1be30baece1d230ec575a7-Abstract-Conference.html)

In [ ]:
adata = sc.read_h5ad("/fse/home/wupengpeng/PertHub/data/sciplex3_biolord.h5ad")
adata

- Skip `normalize_total` and `log1p` if the data is pre-log-normalized; otherwise, apply both.

- Note: Tahoe-100M was pre-trained with `normalize_total=1e4`. You must use the same value for zero-shot prediction to maintain consistency.

In [ ]:
guess_is_lognorm(adata)

# if False, do normalize_total(1e4) and log1p.

- **Gene Alignment and Padding**: After normalize_total and log1p, align genes to the pretrained vocabulary. Retain overlapping genes, pad missing genes with zeros, and ensure the final adata matches the pretrained vocabulary in both gene number and order.

- **Pretrained Model Selection**: Three pretrained models are available: `wpp02/descope-tahoe100m-2k-hvgs`, `wpp02/descope-tahoe100m-5k-hvgs`, and `wpp02/descope-tahoe100m-12059-hvgs`. Select the model that best matches your gene coverage.

In [ ]:
repo_id = "wpp02/descope-tahoe100m-12059-hvgs"
local_dir = repo_id.split("/")[-1]

# Download from huggingface model hub
hf_download(
    repo_id=repo_id,
    repo_type="model",
    local_dir=local_dir
)

# Check gene coverage
genes_in_adata = adata.var_names.tolist()

with open(next(Path(local_dir).glob("*.pkl")), "rb") as f:
    genes_in_vocab = pkl.load(f)

genes_both_in_adata_and_vocab = list(set(genes_in_adata) & set(genes_in_vocab))  # Reliably predicted genes

with open("./reliably_genes.pkl", "wb") as f:
    pkl.dump(genes_both_in_adata_and_vocab, f)  # save reliably predicted genes

print(f"Gene coverage: {len(genes_both_in_adata_and_vocab) / len(genes_in_vocab)}")

# Align
aligner = UniformFeatureForAnnData(
    input_h5ad=adata,
    feature_names_col=None,
    duplicated_features_handling=DuplicatedFeatureHandling.mean_pooling
)
adata_aligned = aligner(genes_in_vocab)
adata_aligned

To ensure consistency with the pre-training dataset (Tahoe-100M), apply the following preprocessing steps:

1. **Control Calibration**: Explicitly set the drug dosage of control cells to 0.

2. **Dose Normalization**: Scale the drug dosages by **dividing by the maximum dosage in the current dataset**.

**[Tips]**: In contrast to the zero-shot approach, dosage normalization based on the Tahoe dataset is not required in this case.

In [ ]:
# Copy essential cell attributes
adata_aligned.obs["cell_type"] = adata.obs["cell_type"].astype(str).str.strip().values  # copy cell_type
adata_aligned.obs["condition"] = adata.obs["condition"].astype(str).str.strip().values  # copy condition
adata_aligned.obs["SMILES"] = adata.obs["SMILES"].astype(str).str.strip().values  # copy SMILES
adata_aligned.obs["split_ood"] = adata.obs["split_ood"].astype(str).str.strip().values  # copy split_ood
adata_aligned.obs["dose"] = adata.obs["dose"].values  # copy dose

adata_aligned.obs.loc[adata_aligned.obs["condition"] == "control", "dose"] = 0  # set the drug dosage of control cells to 0
adata_aligned.obs["dose_norm"] = adata_aligned.obs["dose"] / adata_aligned.obs["dose"].max()  # dose normalization
adata_aligned.obs["drug_and_dose"] = adata_aligned.obs["condition"].astype(str) + "_" + adata_aligned.obs["dose_norm"].astype(str)

In [ ]:
def generate_test_csv_template(
    adata: sc.AnnData, 
    condition_col: str, 
    control_condition: str,
    save_path: str
):
    target_gene, n_cells = [], []
    for cond in adata.obs[condition_col].unique():
        target_gene.append(cond)
        n_cells.append((adata.obs[condition_col] == control_condition).sum())  # consistent with biolord

    df = pd.DataFrame(
        {"target_gene": target_gene, "n_cells": n_cells}
    )
    df.to_csv(save_path, index=False)


# Extract ground_truth
## We uniformly use control cells from the validation set (adata.obs['split_ood'] == 'test')
for ct, ad in tqdm(split_anndata_on_celltype(adata_aligned, celltype_col="cell_type").items()):
    ad = sc.concat(
        [
            ad[(ad.obs["split_ood"] == "ood") & (ad.obs["condition"] != "control")],
            ad[(ad.obs["split_ood"] == "test") & (ad.obs["condition"] == "control")]
        ]
    )
    ad.write_h5ad(f"./{ct}_ground_truth.h5ad")

    # Generate test_csv template
    generate_test_csv_template(
        ad, "drug_and_dose", "control_0.0", f"test_info_{ct}.csv"
    )

In [ ]:
# Extract the training adata and perform tokenization
## We uniformly use control cells from the validation set (adata.obs['split_ood'] == 'test')
for ct, ad in tqdm(split_anndata_on_celltype(adata_aligned, celltype_col="cell_type").items()):
    ad = sc.concat(
        [
            ad[(ad.obs["split_ood"] == "train") & (ad.obs["condition"] != "control")],
            ad[(ad.obs["split_ood"] == "test") & (ad.obs["condition"] == "control")]
        ]
    )
    
    tokenize_adata_to_hf_dataset_for_rna(
        adata=ad,
        cell_line_name=ct,
        target_sum=1e4,
        pert_col="drug_and_dose",
        ctrl_name="control_0.0",
        skip_raw_counts_check=False,
        save_dir=f"./sciplex3_train/{ct}"
    )

- Extract the Uni-Mol2 representations of the drugs.

In [ ]:
drug_smiles_dict = dict(zip(adata_aligned.obs["condition"], adata_aligned.obs["SMILES"]))

clf = UniMolRepr(
    data_type='molecule', 
    remove_hs=False,
    model_name='unimolv2',  # avaliable: unimolv1, unimolv2
    model_size='1.1B',  # work when model_name is unimolv2. avaliable: 84m, 164m, 310m, 570m, 1.1B.
    batch_size=32
)

unimol_repr = clf.get_repr(list(drug_smiles_dict.values()), return_atomic_reprs=True)

# CLS token repr
drug_embed = torch.tensor(unimol_repr['cls_repr'], dtype=torch.float32)

# Apeend drug dose to Uni-Mol2 embedding
unique_dose = adata_aligned.obs["dose_norm"].unique().tolist()
results = {}  # {"drugname_dose": drug_embed_with_dose}
for drug, embed in tqdm(zip(drug_smiles_dict.keys(), drug_embed), total=len(drug_smiles_dict)):
    for d in unique_dose:
        if drug != "control" and d == 0:
            continue  # skip zero dose for non-control drugs
        elif drug == "control" and d != 0:
            continue  # skip non-zero dose for control drug
        else:
            dose_norm = float(d)
            results[f"{drug}_{dose_norm}"] = torch.cat([embed, torch.tensor([dose_norm], dtype=torch.float32)])

torch.save(results, "./drug_dose_embed.pt")